# Demo 2 · Explore an NHL game

**Acquisition, cleaning, exploration and visualisation**

By the end of this notebook, you will be able to inspect raw events, build an
interactive visualisation and discuss its limitations. Code is presented in
small commented steps. Suggested exploration and visualisation time: 25–30 minutes.

## 0 · Install the libraries

**Colab:** upload this notebook, select a CPU runtime and run all.
This cell installs `uv`, then the libraries into the notebook's Python.
`sys.executable` is that Python's path; `subprocess.check_call` runs a command
and stops if it fails. No clone or external Python file is required.
If Colab requests a restart after installation, restart the session and run all again.

**Locally:** run `uv sync` in the repository and use its `.venv`.
This cell installs nothing locally. Colab requires Internet.


In [ ]:
import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = ["pandas>=2.2,<4", "requests>=2.31,<3",
                "plotly>=6,<7", "ipywidgets>=8.1,<9"]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    # Enable interactive sliders in Colab.
    output.enable_custom_widget_manager()


### The data folder
`Path` constructs paths. In Colab we use the current folder.
Locally we locate the repository root to reuse its cache.
JSON lives in `data/raw`; exports go into `data/processed`.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not IN_COLAB:
    # Walk up to the local repository.
    for folder in [ROOT, *ROOT.parents]:
        if (folder / "demo_2").is_dir():
            ROOT = folder
            break
os.chdir(ROOT)
print(ROOT / "data/raw/2025030311.json")

# Demo 2 · From an API to a pandas table

In Demo 1, we started with tables that were ready to use. This time, we will
**get one real NHL game from the internet and build the table ourselves**.
No previous API experience or hockey knowledge is needed. You should be
comfortable running notebook cells and recognizing Python lists and dictionaries.

By the end, you will be able to:

- Explain an API request and response.
- Download one game's play-by-play data using Python.
- Save the original data and load it again without using the internet.
- Turn nested JSON into a pandas DataFrame, inspect it, and retain shots and goals.
- Check missing values and event identifiers before using the table.

**Our pipeline:** request → save raw JSON → load JSON → build a table → check it.

### Before you start

Use the shared course environment: run `uv sync` from the `ift3700-6758` repository
root, then select its `.venv` as your notebook kernel. See the repository README
for setup. This notebook uses **requests** for HTTP and **pandas** for tables;
`json` and `pathlib` come with Python.

Run cells from top to bottom with **Shift+Enter**. If variables get confusing,
choose **Restart Kernel and Run All**. The first run needs internet access.
Later runs reuse a saved copy of this completed game.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

## A. What is an API?

An **API** (Application Programming Interface) is a way for one program to ask
another program for something using agreed rules. A website often returns a page
for a person to read; a web API can return structured data for a program to use.

Here, your Python notebook is the **client**, and the NHL operates the **server**.
Our request asks: “Please send the recorded events for this game.”

```text
Your notebook  ── HTTP GET + a URL ──>  NHL server
Your notebook  <── status + headers + JSON body ──  NHL server
```

**HTTP** is the protocol for these messages. **GET** means retrieve a resource.
**REST** is a style of API design that identifies resources with URLs and uses
HTTP methods to work with them. Each request carries what the server needs to
handle it; the server need not remember an earlier conversation with the client.
We only need GET in this demo. APIs can return formats other than JSON, and not
every API is a REST API.

This public NHL endpoint currently needs **no account or API key**. Other APIs
may require authentication. We are retrieving data, not scraping an HTML page.

### Read the address before writing code

We will use Montréal at Carolina on **May 21, 2026**. An **endpoint** is an API address for a particular resource.

```text
https://api-web.nhle.com/v1/gamecenter/2025030311/play-by-play
        └── server ───┘ └────────── endpoint path ─────────┘
```

| URL part | Meaning here |
|---|---|
| `https://` | HTTP over an encrypted connection |
| `api-web.nhle.com` | The server we contact |
| `/v1` | The API version in the path |
| `/gamecenter/2025030311` | The game resource and its identifier |
| `/play-by-play` | The event data we want |

`2025030311` is a game **identifier**, not a date: `2025` refers to the season's
starting year, `03` to the playoffs, and `0311` identifies the playoff game.
We use a known ID; do not assume every possible number corresponds to a game.

Open the [raw API response](https://api-web.nhle.com/v1/gamecenter/2025030311/play-by-play)
in a browser and compare it with the
[NHL game page](https://www.nhl.com/gamecenter/mtl-vs-car/2026/05/21/2025030311/playbyplay).
Your browser may display the JSON as a long line or a collapsible tree.

**Predict:** which part of the URL would you change to request another game?
Would that require changing the server address?

In [ ]:
game_id = 2025030311
url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
print(url)

## B. Download once and save the raw response

A response has a **status code**, **headers** describing it, and a **body**
containing the data. For a successful request here, expect status `200` and
content type `application/json`.

| Code | Meaning | What to do |
|---|---|---|
| `200` | Request succeeded | Inspect the returned data |
| `404` | Resource not found | Check the endpoint and game ID |
| `403` | Access refused | Check access restrictions; ask the instructor if needed |
| `429` | Too many requests | Stop requesting and wait; respect `Retry-After` if supplied |
| `5xx` | Server problem | Keep your local data and try again later |

The next two cells keep downloads in a local folder. A **cache** is a saved copy
that we can reuse. File paths below are relative to the notebook kernel's working
directory, which can differ between editors. We print the full path to make it
visible. Keep the same working directory when rerunning the notebook.

In [ ]:
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
raw_path = raw_dir / f"{game_id}.json"
print("Raw data file:", raw_path.resolve())

Read the download cell before running it:

1. `exists()` checks whether we already saved this game.
2. `requests.get(...)` sends the GET request. `timeout=30` limits waits for the
   connection and incoming data; it is not a total-download stopwatch.
3. `raise_for_status()` stops on an HTTP error. An error response can also contain
   JSON, so decoding JSON alone does **not** prove that the request succeeded.
4. `.json()` decodes the response into Python objects.
5. `json.dumps(...)` converts those objects to JSON text; `write_text(...)` saves it.

We keep the original fields instead of saving only a filtered table. This lets
us change cleaning decisions later without downloading again. The indentation
makes the saved JSON readable; its whitespace need not match the HTTP response.

In [ ]:
if raw_path.exists():
    print("Already downloaded — reusing", raw_path.name)
else:
    response = requests.get(url, timeout=30)
    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    response.raise_for_status()
    downloaded_game = response.json()

    # Check we received the requested game's event data before saving it.
    assert downloaded_game["id"] == game_id
    assert isinstance(downloaded_game["plays"], list)
    raw_path.write_text(json.dumps(downloaded_game, indent=2), encoding="utf-8")
    print("Saved", raw_path.name)

## C. Load and explore JSON

Loading is a separate step: it reads a file on your computer and makes **no API
request**. `read_text()` reads text; `json.loads()` parses that text into Python
objects. (`loads` means “load from a string”.)

**JSON** (JavaScript Object Notation) is a text format for structured data.
It is not a DataFrame and does not have to look like a table.

| JSON | Python after decoding |
|---|---|
| object: `{"id": 123}` | dictionary (`dict`) |
| array: `[1, 2, 3]` | list (`list`) |
| string / number | `str` / `int` or `float` |
| `true`, `false`, `null` | `True`, `False`, `None` |

In [ ]:
game = json.loads(raw_path.read_text(encoding="utf-8"))
assert game["id"] == game_id
print("Python type:", type(game))
print("Top-level keys:", list(game.keys()))

In [ ]:
print("Game:", game["id"])
print("Date:", game["gameDate"])
print("Match:", game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"])
print("Season:", game["season"])
print("Game type:", game["gameType"])

The whole game is a dictionary. `game["plays"]` is a **list of event dictionaries**.
A play is a recorded event such as a faceoff, shot, goal, penalty, or period start.
It is not necessarily a shot and does not describe every movement on the ice.

Square brackets have two roles below: a string selects a dictionary key, while
an integer selects a list position. Python list positions start at zero.

In [ ]:
plays = game["plays"]
print("Type:", type(plays))
print("Number of events:", len(plays))
print("First event:\n", json.dumps(plays[0], indent=2))

In [ ]:
first_play = plays[0]
print("Event type:", first_play["typeDescKey"])
print("Period:", first_play["periodDescriptor"]["number"])
print("Time in period:", first_play["timeInPeriod"])
print("Optional details:", first_play.get("details", {}))

`periodDescriptor` is a dictionary **inside** an event dictionary. This is nested
data. Events do not all have the same fields: a period start may have no `details`
or coordinates. `get("details", {})` returns an empty dictionary if that key is
absent, so we can inspect it without a `KeyError`. It does not invent coordinates.

**Exercise 1 (2 minutes):** display the second event and print its event type.
What would `plays[1]` mean? What would `game[1]` try to do?

In [ ]:
# Your turn: display the second event and print its event type.

<details>
<summary>Reference answer — open after trying</summary>

```python
print(json.dumps(plays[1], indent=2))
print(plays[1]["typeDescKey"])
```

`plays[1]` selects the second item in a list. `game[1]` looks for the dictionary
key `1`, which this game dictionary does not have.

</details>

## D. From nested events to a DataFrame

Before creating a table, decide what **one row represents**: here, one recorded
event from one game. A plain DataFrame puts nested dictionaries in individual
cells. Let's see that first.

In [ ]:
nested_events = pd.DataFrame(plays)
display(nested_events[["eventId", "typeDescKey", "periodDescriptor"]].head())

`pd.json_normalize` flattens nested dictionaries into columns. For example,
`periodDescriptor.number` becomes a column name. The dot is part of that name,
so select it with `events["periodDescriptor.number"]`.

When a field is absent from an event, pandas marks the corresponding entry as
missing. Flattening changes the **structure**, but it does not decide which
fields or events are useful for our question.

In [ ]:
events = pd.json_normalize(plays)
print("Rows, columns:", events.shape)
print("Columns:", events.columns.tolist())
display(events.head())

In [ ]:
events.info()

In [ ]:
display(events["typeDescKey"].value_counts().rename("event_count").to_frame())
display(events[["details.xCoord", "details.yCoord"]].isna().sum().rename("missing_count"))

**Pause and explain:** why does the table have more rows than the number of goals?
Why might a period-start event have missing coordinates? Would filling those
coordinates with zero preserve the meaning of the data?

The DataFrame index is a row label created by pandas. It is **not** the NHL event
ID. Keep `eventId`, and later combine it with the game ID to identify an event
across several games.

## E. Make a small, useful table

Milestone 1 starts with **shots on goal and goals**. In this feed, `shot-on-goal`
and `goal` are separate event categories. We keep both; a goal is not also a
second `shot-on-goal` row. Missed and blocked shots are other categories that
we leave out of this starter table.

`.isin(...)` builds a Boolean mask; `.loc[...]` selects the matching rows.
`.copy()` gives us a separate table to edit. This is a choice about the question
we want to study, not a claim that the other events are bad data.

In [ ]:
keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])
shots = events.loc[keep].copy()
print("All events:", len(events))
print("Retained shots and goals:", len(shots))
print("Other events left out:", len(events) - len(shots))

Select a few columns and give them readable names. `time_in_period` is the elapsed
clock time **within that period**, not time since the start of the game.
`period_type` keeps regulation, overtime, and shootout distinguishable when you
later inspect other games.

`reindex(columns=...)` also creates missing columns if an optional field is absent
from the entire response. We retain unknown values rather than guessing them.
Game-level metadata is added to every row so the table remains understandable
when we later combine games.

In [ ]:
column_names = {
    "eventId": "event_id",
    "periodDescriptor.number": "period",
    "periodDescriptor.periodType": "period_type",
    "timeInPeriod": "time_in_period",
    "typeDescKey": "event_type",
    "details.eventOwnerTeamId": "team_id",
    "details.xCoord": "x",
    "details.yCoord": "y",
    "details.shotType": "shot_type",
}
shots = shots.reindex(columns=list(column_names)).rename(columns=column_names)
shots["game_id"] = game["id"]
shots["season"] = game["season"]
shots["game_type"] = game["gameType"]
shots["is_goal"] = shots["event_type"].eq("goal")

The event owner identifies the team taking the shot for these retained events.
Use the two team IDs supplied in the game to attach readable abbreviations.
`.map(...)` looks up each ID in a dictionary; an unknown ID stays missing.

`convert_dtypes()` chooses pandas types that can represent missing values.
For example, `Int64` (capital I) stores integers **and** a missing marker, unlike
a plain NumPy integer column. IDs label things; averaging them is meaningless.

In [ ]:
team_names = {
    game["awayTeam"]["id"]: game["awayTeam"]["abbrev"],
    game["homeTeam"]["id"]: game["homeTeam"]["abbrev"],
}
shots["team"] = shots["team_id"].map(team_names)
shots = shots.convert_dtypes()
display(shots.head(10))

### Check before using the table

An `assert` stops execution if its condition is false. These checks ask whether
we accidentally lost rows, lost event identifiers, or created duplicate events.
An event is identified by **both** its game ID and event ID. Do not silently
remove duplicate records before understanding why they are present.

In [ ]:
assert len(events) == len(plays), "Flattening changed the number of events."
assert len(shots) == int(keep.sum()), "Cleaning changed the number of retained events."
assert shots[["game_id", "event_id"]].notna().all().all(), "Missing event identifier."
assert not shots.duplicated(["game_id", "event_id"]).any(), "Duplicate event identifiers."

print("Checks passed.")
display(shots.isna().sum().rename("missing_count").to_frame())

**Missing is not zero.** An unknown coordinate is not the centre of the rink, and
an unknown property is not automatically `False`. Keep those rows in `shots`.
If a later analysis needs both coordinates, make a separate subset and report
how many rows it excludes. For this game, the number can be zero: a check is
still useful when it finds no problem.

In [ ]:
shots_with_coordinates = shots.dropna(subset=["x", "y"])
print("Rows excluded from a coordinate-based analysis:", len(shots) - len(shots_with_coordinates))
print("Rows kept in the full shots table:", len(shots))

### Read a first summary

With Boolean values, `sum()` counts the `True` entries. `size` counts rows.
Here, `shot_events` includes retained non-goal shots **and** goals. This tiny
summary helps us check the data; one game is not enough to rank teams or players.

In [ ]:
summary = shots.groupby("team", dropna=False).agg(
    shot_events=("event_id", "size"),
    goals=("is_goal", "sum"),
)
display(summary)

**Exercise 2 (3 minutes):** make a table containing only first-period goals.
Show `team`, `time_in_period`, and `shot_type`. How many rows does it have?

Hint: combine two Boolean conditions with `&`, and put parentheses around each.

In [ ]:
# Your turn: filter shots, then select the three requested columns.

<details>
<summary>Reference answer — open after trying</summary>

```python
first_period_goals = shots.loc[
    (shots["period"] == 1) & shots["is_goal"],
    ["team", "time_in_period", "shot_type"],
]
display(first_period_goals)
print("Number of first-period goals:", len(first_period_goals))
```

</details>

## F. Save the derived table and connect the steps

The **raw JSON** preserves the source fields. A **processed CSV** stores our chosen
rows and columns. Keep both: you can rebuild the CSV from the JSON if your cleaning
rules change. `index=False` avoids writing pandas' row labels as an extra column.

CSV is convenient for inspection and sharing, but it does not preserve all pandas
types. When loading it later, check types again, especially identifiers and missing
values. The raw JSON remains the input to our cleaning process.

In [ ]:
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
csv_path = processed_dir / f"{game_id}_shots.csv"
shots.to_csv(csv_path, index=False)
print("Saved processed table:", csv_path.resolve())

## Objective 2 · inspect before plotting
Use **part 1's `shots` table** directly, without cleaning it again.
`events` contains all events; `shots` contains shots on goal **including goals**.
Start by counting event types. What does one row represent?

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from ipywidgets import interact, IntSlider

LANG = 'en'
pio.renderers.default = "colab" if IN_COLAB else "plotly_mimetype+notebook"
teams = [game["awayTeam"]["abbrev"], game["homeTeam"]["abbrev"]]
colors = dict(zip(teams, ["#2364ce", "#e25c45"]))

display(events["typeDescKey"].value_counts().rename("count").to_frame())
if LANG == "fr":
    print(f"{len(events)} événements → {len(shots)} tirs cadrés, buts inclus")
else:
    print(f"{len(events)} events → {len(shots)} shots on goal, goals included")

### A small reusable function: draw the rink
A function groups instructions. This one adds a rectangle and three lines to a
Plotly figure, then returns it. Fixed axes preserve the proportions.
Coordinates are in feet. This is a schematic, not a hockey-specific package.

In [ ]:
def draw_rink(fig):
    # A 200-foot by 85-foot rectangle.
    fig.add_shape(type="rect", x0=-100, x1=100, y0=-42.5, y1=42.5,
                  line_color="#7f95aa")
    for x, colour in [(-25, "#9bc2e6"), (0, "#e6a0a0"), (25, "#9bc2e6")]:
        fig.add_vline(x=x, line_color=colour)
    fig.update_xaxes(range=[-103, 103], title="x (ft)", constrain="domain")
    fig.update_yaxes(range=[-46, 46], title="y (ft)", scaleanchor="x", scaleratio=1)
    fig.update_layout(template="plotly_white", height=460,
                      margin=dict(l=55, r=25, t=55, b=50))
    return fig

### A slider becomes a function argument
`@interact` calls the function whenever the slider changes.
`i` is a **list position**, not the NHL event identifier.
The table below lists goal and period-start positions to try.
Without coordinates, show the JSON and an empty rink, never a fake point at `(0, 0)`.

In [ ]:
examples = events["typeDescKey"].isin(["goal", "period-start"])
display(events.loc[examples, ["eventId", "typeDescKey"]])

@interact(i=IntSlider(min=0, max=len(game["plays"])-1, continuous_update=False))
def inspect_event(i):
    play = game["plays"][i]
    print(json.dumps(play, indent=2, ensure_ascii=False))
    details = play.get("details", {})
    x = details.get("xCoord")
    y = details.get("yCoord")
    fig = go.Figure()
    # Plot only if both coordinates exist.
    if x is not None and y is not None:
        fig.add_scatter(x=[x], y=[y], mode="markers", marker_size=18)
    fig.update_layout(title=f"eventId={play['eventId']} · {play['typeDescKey']}")
    draw_rink(fig).show()

## Objective 3 · build a clock
`mm:ss` is elapsed time within the period. For periods 1–3:
`minute = (period - 1) × 20 + mm + ss / 60`.
A copy named `timed` excludes overtime and shootouts; `shots` stays intact.

In [ ]:
regular = (shots["period_type"] == "REG") & shots["period"].between(1, 3)
timed = shots.loc[regular].copy()
clock = timed["time_in_period"].str.split(":", expand=True).astype(float)
timed["minute"] = (timed["period"] - 1) * 20 + clock[0] + clock[1] / 60
timed = timed.sort_values(["minute", "event_id"])
label = "Tirs exclus du replay :" if LANG == "fr" else "Shots excluded from replay:"
print(label, len(shots) - len(timed))
display(timed[["period", "time_in_period", "minute", "team"]].head())

### The game in one minute
The ▶ button advances the slider. At each minute, filter shots that have already happened.
`color` identifies the team; `symbol` distinguishes goals (stars).
Pause at minute 20 and predict what comes next. This is not puck tracking.

In [ ]:
import ipywidgets as widgets

# Link the play button to the slider.
play = widgets.Play(min=0, max=60, interval=500)
minute_slider = IntSlider(min=0, max=60, continuous_update=False)
widgets.jslink((play, "value"), (minute_slider, "value"))
display(play)

@interact(minute=minute_slider)
def replay_at(minute):
    visible = timed.loc[timed["minute"] <= minute].dropna(subset=["x", "y"])
    title = f"Minute {minute} · {len(visible)} tirs" if LANG == "fr" else f"Minute {minute} · {len(visible)} shots"
    fig = px.scatter(visible, x="x", y="y", color="team", symbol="is_goal",
                     color_discrete_map=colors, symbol_map={False: "circle", True: "star"},
                     category_orders={"team": teams},
                     hover_data=["period", "time_in_period"], title=title)
    fig.update_traces(marker_size=13, marker_opacity=0.85)
    draw_rink(fig).show()

### From animation to analysis: cumulative shots
`cumcount() + 1` numbers each team's shots in chronological order.
A step curve does not suggest shots between events.
A steep slope means many shots, not high possession.

In [ ]:
timed["cumulative_shots"] = timed.groupby("team").cumcount() + 1
fig = go.Figure()
for team, group in timed.groupby("team"):
    # Include the start and end of the game.
    minutes = [0] + group["minute"].tolist() + [60]
    totals = [0] + group["cumulative_shots"].tolist() + [len(group)]
    fig.add_scatter(x=minutes, y=totals, name=team, mode="lines",
                    line=dict(shape="hv", color=colors[team]))
    goals = group.loc[group["is_goal"]]
    fig.add_scatter(x=goals["minute"], y=goals["cumulative_shots"],
                    mode="markers", showlegend=False,
                    marker=dict(symbol="star", size=14, color=colors[team]),
                    hovertemplate=team + " · goal<extra></extra>")
for minute in [20, 40]:
    fig.add_vline(x=minute, line_dash="dot")
y_title = "Tirs cadrés cumulés" if LANG == "fr" else "Cumulative shots on goal"
title = "Quand les tirs s'accumulent-ils ?" if LANG == "fr" else "When do shots accumulate?"
fig.update_layout(title=title, xaxis_title="Minutes", yaxis_title=y_title,
                  template="plotly_white", hovermode="x unified")
fig.show()

### Your turn · 3 minutes
Count each team's shots between minutes 15 and 20.

In [ ]:
window = timed.loc[timed["minute"].between(15, 20)]
# Complete here.

<details><summary>Reference answer</summary>

```python
display(window.groupby("team").size())
```

</details>

## Export and takeaway
`fig` is the last main figure. Its HTML includes Plotly and opens without a Python kernel.
Widgets need an active kernel. In Colab, download the export through the Files panel.
One game illustrates the method; the milestone requires multiple games/seasons and hourly excess maps.
**Lecture 6:** slides 5 (explore/present), 9 (question first), 20–21 (one question per figure), 47 (interactivity and uncertainty).

In [ ]:
output_dir = ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
target = output_dir / "figure_b_en.html"
fig.write_html(target, include_plotlyjs=True)
print(target)